In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.graph_objs as go
import plotly.offline as py
import plotly.express as px

#Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
#By Ranamalla Nithin Reddy https://www.kaggle.com/code/nithinreddy90/chatpgpt-prompts

from transformers import AutoTokenizer

df = pd.read_csv('data/train.csv')

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Preprocess the data
df.drop_duplicates(inplace=True)
df.dropna(subset=['output', 'instruction'], inplace=True)

# Tokenize prompts and actions
df['instruction_tokens'] = df['instruction'].apply(lambda x: len(tokenizer.tokenize(x)))
df['output_tokens'] = df['output'].apply(lambda x: len(tokenizer.tokenize(x)))

# Display the preprocessed and tokenized dataframe
print(df.head())

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1087 > 1024). Running this sequence through the model will result in indexing errors


  input                                             output  \
0   NaN  To find the probability of the spinner landing...   
1   NaN  I need to choose 6 people out of 14, and the o...   
2   NaN  First we count the number of all 4-letter word...   
3   NaN  She can do this if and only if at least one of...   
4   NaN  Think of the problem as a sequence of H's and ...   

                                         instruction    data_source  \
0  A board game spinner is divided into three par...  MATH/PRM-800K   
1  My school's math club has 6 boys and 8 girls. ...  MATH/PRM-800K   
2  How many 4-letter words with at least one cons...  MATH/PRM-800K   
3  Melinda will roll two standard six-sided dice ...  MATH/PRM-800K   
4  Let $p$ be the probability that, in the proces...  MATH/PRM-800K   

   instruction_tokens  output_tokens  
0                  86            215  
1                  49             92  
2                  73            170  
3                  72             90  
4    

In [3]:
# --- [CELL 2]: ---
# cell_state: edited
# execution_status: {'status': 'error', 'done': True, 'execution_count': 3}
# === BEFORE (original) ===
# import pandas as pd
# import torch
# from transformers import GPT2LMHeadModel, GPT2Tokenizer
# 
# # Load pre-trained model and tokenizer
# model_name = "gpt2"
# model = GPT2LMHeadModel.from_pretrained(model_name)
# tokenizer = GPT2Tokenizer.from_pretrained(model_name)
# 
# # Load prompts from DataFrame
# # (Assuming you've loaded your prompts into a DataFrame named df_prompts)
# generated_responses = []
# 
# for index, row in df.iterrows():
#     prompt = row['instruction']
#     input_ids = tokenizer.encode(prompt, return_tensors="pt")
#     
#     # Generate response
#     with torch.no_grad():
#         output = model.generate(
#             input_ids,
#             max_length=input_ids.size(1) + 50,  # Adjust the additional tokens as needed
#             num_return_sequences=1,
#             pad_token_id=tokenizer.eos_token_id,
#             attention_mask=input_ids.ne(tokenizer.pad_token_id)
#         )
#     
#     # Pad the generated sequence
#     padded_output = output[:, input_ids.size(1):]
#     
#     response = tokenizer.decode(padded_output[0], skip_special_tokens=True)
#     generated_responses.append(response)

# === AFTER (edited) ===
import pandas as pd
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer


model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 has no pad token by default; use EOS as PAD to avoid attention mask/pad-token issues.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# Keep model in eval mode for generation.
model.eval()

generated_responses = []

for index, row in df.iterrows():
    prompt = str(row['instruction'])
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    # Build a valid attention mask (all ones since no padding in a single encoded prompt)
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=input_ids.size(1) + 50,
            num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id,
        )

    padded_output = output[:, input_ids.size(1):]
    response = tokenizer.decode(padded_output[0], skip_special_tokens=True)
    generated_responses.append(response)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1087 > 1024). Running this sequence through the model will result in indexing errors


IndexError: index out of range in self

In [4]:
assert tokenizer.pad_token is not None, "pad_token should be set for GPT-2"
assert tokenizer.pad_token == tokenizer.eos_token, "pad_token should be eos_token for this fix"
assert tokenizer.pad_token_id == tokenizer.eos_token_id, "pad/eos token ids should match"

# generation artifact checks
assert isinstance(generated_responses, list) and len(generated_responses) == 2
assert all(isinstance(r, str) and len(r.strip()) > 0 for r in generated_responses)

AssertionError: 